# Lab 04: The Planning Pattern for Adaptive Timeline Reconstruction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danjethh/sectratek-AI-Lab/blob/main/lab04_planning_pattern/03_lab_notebook.ipynb)

**Course:** AI for Cybersecurity and Digital Forensics  
**Instructor:** Daniel Kwaku Ntiamoah Addai, Mobile Threat Research Lead, SectraTEK Lab  
**Runs in:** Google Colab or any hosted Jupyter. Nothing is installed on your own computer.

---

Lab 3's ReAct loop decided the next step only after seeing the last result. **Planning** starts
differently: for a larger question with several connected parts, the model first lays out an ordered
plan, then **revises it** when a new observation breaks an assumption.

You will reconstruct a phone's communication timeline during a 30-minute window when it went missing.
You build an initial plan from part of the evidence, then a withheld **network record** arrives and
forces a replan: it turns out the phone was offline, so a WhatsApp image attachment inside the window
cannot be called a confirmed delivery. You walk the plan-replan loop by hand (Part A), then run it
through a packaged `PlanningAgent` (Part B).

## Part 0: One-Time Setup

Run these cells in order. They install one library, connect you to a **free hosted model** (Groq),
write the case files, and load the planning framework. No local install is required.

### Step 0.1: Install the client library

**Purpose:** install the `openai` client, the one small library that lets the notebook talk to the model (Groq uses the same interface).  
**What you'll see:** a short pip log. Warnings are normal.

In [ ]:
# Purpose: install the one library this notebook needs, inside the hosted session only.
%pip install -q openai

### Step 0.2: Connect to a free hosted model (Groq)

**Purpose:** get a free **API key** (a private password that lets the notebook use the model) and build a client.

1. Open **https://console.groq.com/keys**, sign in, and create a key.
2. Run the cell and paste the key when asked. It is hidden and is **not saved** into the notebook.

**What you'll see:** `Model OK: ...` once the connection works.

In [ ]:
# Purpose: collect the Groq key privately and build an OpenAI-compatible client pointed at Groq.
import os        # os: lets the notebook read saved settings, such as where your key is kept
import getpass   # getpass: lets you type your secret key without it appearing on screen
from openai import OpenAI  # OpenAI: the client that carries your messages to the model (Groq speaks the same language)

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass('Paste your Groq API key (hidden): ').strip()

BASE_URL = 'https://api.groq.com/openai/v1'
MODEL = 'llama-3.3-70b-versatile'  # strong at structured planning; free on Groq

client = OpenAI(base_url=BASE_URL, api_key=os.environ['GROQ_API_KEY'])

try:
    probe = client.chat.completions.create(
        model=MODEL, messages=[{'role': 'user', 'content': 'Reply with one word: ready'}])
    print('Model OK:', MODEL, '->', probe.choices[0].message.content.strip())
except Exception as err:
    print('Could not use', MODEL, '->', err)
    print('\nModels available on your key:')
    try:
        for m in client.models.list().data:
            print('  -', m.id)
    except Exception as err2:
        print('  (could not list models:', err2, ')')
    print('\nSet MODEL above to one of these and re-run.')

### Step 0.3: Write the synthetic case files

**Purpose:** create a local `data/` folder with the six artifacts for case `PL-2026-011`. These are
synthetic and identical to the repository's copies.  
**What you'll see:** the files written. Existing files are not overwritten.

In [ ]:
# Purpose: create the case files on disk so the notebook has everything it needs, with no download.
from pathlib import Path  # Path: a readable way to refer to files and folders by name

ARTIFACTS = {'artifact_manifest.json': '{\n  "case_id": "PL-2026-011",\n  "lab": "Lab 4 - Planning Pattern",\n  "device": {\n    "platform": "Android",\n    "model": "Pixel 8",\n    "os_version": "Android 14"\n  },\n  "acquisition": {\n    "type": "full_file_system",\n    "tool": "Cellebrite UFED 7.70",\n    "acquired_at_utc": "2026-02-12T13:11:42Z",\n    "original_image_sha256": "4bc2ff573aa8bf3d6b0fb7a81341a7ef34df7b64c1d50368f6dd80e38f4a95ce"\n  },\n  "analysis_timezone": "America/New_York",\n  "incident_window_utc": {\n    "start": "2026-02-11T20:55:00Z",\n    "end": "2026-02-11T21:25:00Z"\n  },\n  "note": "Synthetic classroom dataset for planning-pattern demo"\n}\n', 'unlock_events.csv': 'timestamp_utc,event,source\n2026-02-11T20:55:03Z,device_unlocked,keyguard_log\n2026-02-11T21:25:31Z,device_locked,keyguard_log\n', 'call_log.csv': 'timestamp_utc,direction,phone_number,duration_seconds,call_type\n2026-02-11T21:08:14Z,outgoing,+1-555-0184,42,cellular_voice\n', 'whatsapp_events.csv': 'timestamp_utc,event,chat_id,details\n2026-02-11T21:12:07Z,chat_opened,client_thread_18,"Conversation opened from recent chats"\n2026-02-11T21:13:12Z,message_deleted,client_thread_18,"Deleted outbound message with image attachment placeholder"\n2026-02-11T21:13:19Z,attachment_event,client_thread_18,"Image attachment record observed in app cache metadata"\n', 'network_status.csv': 'start_utc,end_utc,network_state,notes\n2026-02-11T20:55:00Z,2026-02-11T21:10:00Z,online,"Mobile data available"\n2026-02-11T21:10:00Z,2026-02-11T21:26:59Z,offline,"No mobile data or Wi-Fi connectivity recorded"\n2026-02-11T21:27:04Z,2026-02-11T21:40:00Z,online,"Mobile data restored"\n', 'chain_of_custody.csv': 'timestamp_utc,handler,action,notes\n2026-02-12T13:11:42Z,Investigator_A,acquired_image,"UFED acquisition to encrypted evidence drive"\n2026-02-12T13:38:57Z,Investigator_A,computed_hash,"SHA-256 verified against acquisition log"\n2026-02-13T08:46:12Z,Analyst_B,created_working_copy,"Exported access, call, app, and network artifacts"\n'}

data_dir = Path('data'); data_dir.mkdir(exist_ok=True)
for name, text in ARTIFACTS.items():
    fp = data_dir / name
    if not fp.exists():
        fp.write_text(text)
    print(('wrote ' if fp.stat().st_size else 'exists ') + str(fp))

### Step 0.4: Load the planning framework

**Purpose:** load the packaged `PlanningAgent`. This mirrors
`src/agentic_patterns/planning_pattern/` in the repository and is adapted from The Neural Maze's
agentic-patterns-course (MIT License).  
**What you'll see:** nothing printed; it just defines the class used in Part B.

In [ ]:
# Purpose: the packaged PlanningAgent (build_initial_plan, revise_plan, run). Same code as the repo.
# Adapted from The Neural Maze agentic-patterns-course (MIT License), (c) 2024 The Neural Maze.

# ----------------------------------------------------------------- chat history
def build_prompt_structure(prompt: str, role: str, tag: str = "") -> dict:
    if tag:
        prompt = f"<{tag}>{prompt}</{tag}>"
    return {"role": role, "content": prompt}


def update_chat_history(history: list, msg: str, role: str):
    history.append(build_prompt_structure(prompt=msg, role=role))


def completions_create(client, messages, model) -> str:
    response = client.chat.completions.create(messages=list(messages), model=model)
    return str(response.choices[0].message.content)


class ChatHistory(list):
    def __init__(self, messages=None, total_length=-1):
        super().__init__(messages or [])
        self.total_length = total_length

    def append(self, msg):
        if len(self) == self.total_length:
            self.pop(0)
        super().append(msg)


class FixedPrefixChatHistory(ChatHistory):
    """Keeps an initial prefix pinned (system + original task) while newer messages rotate."""

    def __init__(self, messages=None, total_length=-1, pinned_prefix_len=1):
        super().__init__(messages, total_length)
        self.pinned_prefix_len = pinned_prefix_len

    def append(self, msg):
        if self.total_length != -1 and len(self) == self.total_length:
            self.pop(self.pinned_prefix_len)
        list.append(self, msg)


# ----------------------------------------------------------------- the planning agent
PLANNING_SYSTEM_PROMPT = """
You are a digital forensics planning agent.

Your job is to break a forensic question into an ordered investigation plan, identify what evidence is needed,
and revise the plan when new observations expose a missing dependency, contradiction, or timing conflict.

When building an initial plan:
- state the investigation goal in plain language
- produce an ordered step-by-step plan
- note the evidence needed for each major step
- identify clear replanning triggers

When revising a plan:
- preserve still-useful steps
- reorder steps when new observations change dependencies
- explain how the new observations change the investigation path
- keep the revised plan concise and evidence-focused

Always avoid unsupported conclusions. Treat planning as an aid to the student, not as final decision authority.
"""


class PlanningAgent:
    """A planning-first agent for building and revising investigation plans.

    Entry points:
    - build_initial_plan(user_msg)            -> the first ordered plan
    - revise_plan(user_msg, plan, observations) -> a plan updated with new evidence
    - run(user_msg, max_replans)              -> plan, self-review, and improve until [[OK]]
    """

    def __init__(self, model="llama-3.3-70b-versatile", system_prompt="", client=None):
        if client is None:
            raise ValueError("PlanningAgent requires a `client` (chat.completions.create).")
        self.client = client
        self.model = model
        self.system_prompt = system_prompt

    def _build_system_prompt(self) -> str:
        return (self.system_prompt + "\n" + PLANNING_SYSTEM_PROMPT).strip()

    def _complete(self, user_msg: str) -> str:
        history = [
            build_prompt_structure(self._build_system_prompt(), "system"),
            build_prompt_structure(user_msg, "user"),
        ]
        return completions_create(self.client, history, self.model)

    def build_initial_plan(self, user_msg: str) -> str:
        request = (
            "Build an initial investigation plan for the following forensic task.\n\n"
            "Return:\n1. Investigation goal\n2. Ordered steps\n3. Evidence needed for each major step\n"
            "4. Replanning triggers\n\n"
            f"Task:\n{user_msg}"
        )
        return self._complete(request)

    def revise_plan(self, user_msg: str, current_plan: str, observations: str) -> str:
        request = (
            "Revise the current forensic investigation plan using the new observations below.\n\n"
            "Return:\n1. What changed\n2. Revised ordered steps\n3. Updated evidence priorities\n"
            "4. Remaining replanning triggers\n\n"
            f"Original task:\n{user_msg}\n\n"
            f"Current plan:\n{current_plan}\n\n"
            f"New observations:\n{observations}"
        )
        return self._complete(request)

    def run(self, user_msg: str, max_replans: int = 3) -> str:
        history = FixedPrefixChatHistory(
            [
                build_prompt_structure(self._build_system_prompt(), "system"),
                build_prompt_structure(user_msg, "user"),
            ],
            total_length=max(6, (max_replans * 2) + 2),
            pinned_prefix_len=2,
        )

        plan = self.build_initial_plan(user_msg)
        update_chat_history(history, plan, "assistant")

        for _ in range(max_replans):
            review_prompt = (
                "Review the current investigation plan.\n"
                "If the plan is already well ordered, evidence-focused, and includes clear replanning triggers, "
                "reply exactly [[OK]].\n"
                "Otherwise, identify the most important missing dependency, contradiction, or observation "
                "that should trigger replanning."
            )
            update_chat_history(history, review_prompt, "user")
            review = completions_create(self.client, history, self.model)
            update_chat_history(history, review, "assistant")

            if "[[OK]]" in review:
                print("Plan accepted by self-review. Stopping.")
                break

            revision_request = (
                "Revise the current plan using the following review notes.\n\n"
                f"Current plan:\n{plan}\n\n"
                f"Review notes:\n{review}"
            )
            update_chat_history(history, revision_request, "user")
            plan = completions_create(self.client, history, self.model)
            update_chat_history(history, plan, "assistant")

        return plan

## The Case in One Minute

**Case `PL-2026-011`** - a caseworker's Google Pixel 8 went missing for about 30 minutes during
evening field visits. You must reconstruct what communication happened inside that window.

| UTC time | What the records show |
|---|---|
| 20:55:03 | device unlocked (start of the missing period) |
| 21:08:14 | outgoing call to +1-555-0184, 42 seconds |
| 21:10:00 | mobile data drops offline |
| 21:12:07 | WhatsApp chat opened |
| 21:13:12 | a message with an image attachment is recorded, then deleted |
| 21:25:31 | device locked (end of the missing period) |
| 21:27:04 | mobile data restored (after the window) |

The call and the WhatsApp activity are inside the window. But the phone was **offline** from 21:10 to
21:26, so the image attachment at 21:13 cannot be called a confirmed delivery. Catching that is the
whole point of replanning.

## Planning versus ReAct

| | **ReAct** (Lab 3) | **Planning** (this lab) |
|---|---|---|
| Starts with | the next single step | an ordered plan for the whole question |
| Good for | one check at a time | several connected, dependent steps |
| Adapts by | choosing the next action | revising the remaining plan |

Think of organizing an event: *estimate guests -> pick a date -> book a room -> order food*. If the
room is taken, a planner reorders the remaining steps instead of charging ahead. Here, the "room is
taken" moment is the network record showing the phone was offline.

## Part A: Walk the Plan-Replan Loop by Hand

You build an initial plan from **part** of the evidence, then a new observation arrives and you revise.

### Step A.1: Build the partial and full evidence bundles

**Purpose:** the initial plan deliberately sees everything **except** the network record. The network
record is held back as a `new_observation` that arrives after the first plan, exactly as a late-
discovered artifact would in a real case.  
**Output:** `partial_artifact_bundle`, `new_observation`, and `full_artifact_bundle`.

In [ ]:
# Purpose: build the partial bundle (no network), the withheld observation, and the full bundle.
from pathlib import Path
from IPython.display import Markdown, display  # show nicely formatted text (headings, bold) in the notebook

data_dir = Path('data')

def excerpt(filename, n_lines=8):
    lines = (data_dir / filename).read_text().strip().splitlines()
    return '\n'.join(lines[:n_lines])

# The initial bundle intentionally leaves out network_status.csv.
partial_artifact_bundle = f'''
artifact_manifest.json
{(data_dir / 'artifact_manifest.json').read_text()}

unlock_events.csv
{excerpt('unlock_events.csv')}

call_log.csv
{excerpt('call_log.csv')}

whatsapp_events.csv
{excerpt('whatsapp_events.csv')}

chain_of_custody.csv
{excerpt('chain_of_custody.csv')}
'''

# The late-arriving observation that forces a replan.
new_observation = f'''
network_status.csv
{excerpt('network_status.csv')}
'''

full_artifact_bundle = partial_artifact_bundle + '\n\n' + new_observation
print('Partial bundle omits network_status.csv; new_observation supplies it later.')

### Step A.2: Build the initial investigation plan

**Purpose:** ask the model for an ordered plan from the partial bundle. Because the network record is
still hidden, a good plan names **replanning triggers** instead of pretending it is complete.  
**What you'll see:** a goal, ordered steps, the evidence each needs, and triggers that would change it.

In [ ]:
# Purpose: produce the first ordered plan from the partial evidence.
PLANNING_PROMPT = (
    'You are a digital forensics planning assistant. Build ordered investigation plans, update them when new observations expose missing dependencies, and avoid unsupported conclusions.')

case_question = (
    'Reconstruct the communication timeline during the incident window and determine which supported events occurred inside it. How should the timeline change if a network-status record is added later?')

initial_plan_request = (
    'Build an initial investigation plan for this case.\n'
    'Return: (1) investigation goal, (2) ordered steps, (3) evidence needed, (4) replanning triggers.\n\n'
    f'Task:\n{case_question}\n\nArtifacts:\n{partial_artifact_bundle}')

initial_plan = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'system', 'content': PLANNING_PROMPT},
              {'role': 'user', 'content': initial_plan_request}],
).choices[0].message.content
display(Markdown('### Initial Investigation Plan\n\n' + initial_plan))

### Step A.3: Replan after the network observation

**Purpose:** the network record shows the phone was offline from 21:10 to 21:26. That changes what the
21:13 WhatsApp attachment can support. Ask the model to **revise** its plan, not start over.  
**What you'll see:** what changed, the revised steps, and why the delivery claim must now be bounded.

In [ ]:
# Purpose: revise the plan using the late-arriving network observation.
revised_plan_request = (
    'Revise the current investigation plan using the new observation below.\n'
    'Return: (1) what changed, (2) revised ordered steps, (3) updated evidence priorities, (4) remaining replanning triggers.\n\n'
    f'Original task:\n{case_question}\n\nCurrent plan:\n{initial_plan}\n\n'
    f'New observation:\n{new_observation}')

revised_plan = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'system', 'content': PLANNING_PROMPT},
              {'role': 'user', 'content': revised_plan_request}],
).choices[0].message.content
display(Markdown('### Observation-Driven Replan\n\n' + revised_plan))

### Step A.4: Write the final, evidence-bounded timeline

**Purpose:** with the revised plan and the full evidence, write a short timeline report. Separate
in-window activity from anything the offline gap leaves unconfirmed.

In [ ]:
# Purpose: produce the final timeline conclusion from the revised plan and full evidence.
final_report_request = (
    'Using the revised plan and full artifact set, produce a short evidence-cited timeline conclusion.\n'
    'Return: (1) reconstructed timeline, (2) timing conclusion, (3) evidence mapping, (4) what remains uncertain.\n\n'
    f'Revised plan:\n{revised_plan}\n\nArtifacts:\n{full_artifact_bundle}')

final_timeline = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'system', 'content': PLANNING_PROMPT},
              {'role': 'user', 'content': final_report_request}],
).choices[0].message.content
display(Markdown('### Reconstructed Timeline and Timing Conclusion\n\n' + final_timeline))

## Part B: Run the Same Workflow with `PlanningAgent`

The packaged agent wraps the same three moves: build a plan, revise it with a new observation, and
(in `run`) self-review and improve until the plan is sound. Compare it with your manual version.

### Step B.1: Build and revise through the agent

**Purpose:** call `build_initial_plan` on the partial bundle, then `revise_plan` with the network
observation. Check that the agent's revision is driven by the evidence, not by restarting.

In [ ]:
# Purpose: run the agent's build-then-revise over the same partial bundle and network observation.
planning_agent = PlanningAgent(client=client, model=MODEL)

agent_task = f'{case_question}\n\nArtifacts:\n{partial_artifact_bundle}'
packaged_initial_plan = planning_agent.build_initial_plan(agent_task)
display(Markdown('### PlanningAgent Initial Plan\n\n' + packaged_initial_plan))

packaged_revised_plan = planning_agent.revise_plan(
    user_msg=agent_task, current_plan=packaged_initial_plan, observations=new_observation)
display(Markdown('### PlanningAgent Revised Plan\n\n' + packaged_revised_plan))

### Step B.2: Run the full self-reviewing workflow

**Purpose:** `run` builds a plan, then critiques and improves it until the review says `[[OK]]` or a
replan limit is reached. The `max_replans` cap keeps a free-tier key from being overused.  
**What you'll see:** a final plan that has survived the agent's own review.

In [ ]:
# Purpose: let the agent plan and self-review over the full evidence bundle.
packaged_final = planning_agent.run(
    user_msg=f'{case_question}\n\nArtifacts:\n{full_artifact_bundle}', max_replans=2)
display(Markdown('### PlanningAgent Final Plan\n\n' + packaged_final))

## Your Timeline and Decision

Write your own answer below, based on the records and the plans above.

1. **Reconstructed timeline:** the key events, in order, with UTC times.
2. **Timing conclusion:** which events are inside the 20:55 to 21:25 window.
3. **Evidence mapping:** which artifact supports each event.
4. **Remaining uncertainty:** what the offline gap (21:10 to 21:26) leaves unconfirmed, including the
   WhatsApp image attachment.

In [ ]:
answer = {
    'timeline': ['20:55:03 unlocked', '21:08:14 outgoing call (42s)', '21:10 offline',
                 '21:12-21:13 WhatsApp chat + attachment', '21:25:31 locked', '21:27 restored'],
    'in_window': ['...'],
    'evidence_mapping': {'call': 'call_log.csv', 'whatsapp': 'whatsapp_events.csv', 'offline': 'network_status.csv'},
    'uncertainty': 'Device offline 21:10-21:26, so the 21:13 image attachment is not a confirmed delivery.',
}
for k, v in answer.items():
    print(k, '->', v)

### Compare the two runs

1. Did your manual replan and `PlanningAgent`'s replan reach the same bounded conclusion?
2. Did either version keep the WhatsApp attachment as an **attempt** rather than a delivery?
3. What single observation did the most to change the plan, and why?

## Submit

Save the notebook with Part A, Part B, and your answer filled in.

- In Colab: **File -> Download -> Download .ipynb**, or **File -> Save a copy in Drive**.

---

*Case data and the lab concept are adapted from*
*[frankwxu/agentic-AI4-forensics](https://github.com/frankwxu/agentic-AI4-forensics). The planning
framework is adapted from [The Neural Maze's agentic-patterns-course](https://github.com/neural-maze/agentic-patterns-course) (MIT License).*